# Prepare Women's Fashion Catalog

Prepare product fields, create `general_category`, and generate embeddings.

In [2]:
import sys
import pandas as pd
import sys
sys.path.insert(0, "../..")

from Shared.product_embedding import create_product_embeddings

products = pd.read_csv("./Raw CSV/products.csv", encoding="utf-8-sig")
images = pd.read_csv("./Raw CSV/images.csv", encoding="utf-8-sig")
variants = pd.read_csv(
    "./Raw CSV/variants.csv",
    encoding="utf-8-sig",
    dtype={"sku": "string"},
)

## Prepare products

In [3]:
clean_products = products.copy()
clean_products["review_count"] = pd.to_numeric(clean_products["review_count"], errors="coerce").astype("Int64")
clean_products["review_rating"] = pd.to_numeric(clean_products["review_rating"], errors="coerce")

fields = {
    "Title": "title",
    "Product type": "product_type",
    "Category": "category_full_path",
    "Tags": "tags",
    "Description": "description",
}
clean_products["search_text"] = clean_products.apply(
    lambda row: "\n".join(
        f"{label}: {row[column] if pd.notna(row[column]) else ''}"
        for label, column in fields.items()
    ),
    axis=1,
)

## Create general category

In [4]:
levels = clean_products["category_full_path"].str.split(">")
level_2 = levels.str[1].str.strip()
level_3 = levels.str[2].str.strip()

is_clothing = level_2.eq("Clothing")
clothing_category = level_3.fillna("Other")
category_counts = clothing_category[is_clothing].value_counts()
small_category = clothing_category.map(category_counts).lt(45)

clean_products["general_category"] = level_2
clean_products.loc[is_clothing, "general_category"] = clothing_category.mask(
    small_category,
    "Other",
)

clean_products["general_category"].value_counts()

general_category
Clothing Tops                720
Jewelry                      361
Dresses                      224
Clothing Accessories         218
Sleepwear & Loungewear       200
Pants                        165
Shorts                       162
One-Pieces                   151
Handbags, Wallets & Cases    145
Swimwear                     121
Other                        111
Outerwear                     79
Shoes                         52
Skirts                        49
Name: count, dtype: int64

## Generate product embeddings

In [5]:
vectors = create_product_embeddings(clean_products)
clean_products["embedding"] = [
    f"[{','.join(map(str, vector))}]"
    for vector in vectors
]

Embedded 100/2,766
Embedded 200/2,766
Embedded 300/2,766
Embedded 400/2,766
Embedded 500/2,766
Embedded 600/2,766
Embedded 700/2,766
Embedded 800/2,766
Embedded 900/2,766
Embedded 1,000/2,766
Embedded 1,100/2,766
Embedded 1,200/2,766
Embedded 1,300/2,766
Embedded 1,400/2,766
Embedded 1,500/2,766
Embedded 1,600/2,766
Embedded 1,700/2,766
Embedded 1,800/2,766
Embedded 1,900/2,766
Embedded 2,000/2,766
Embedded 2,100/2,766
Embedded 2,200/2,766
Embedded 2,300/2,766
Embedded 2,400/2,766
Embedded 2,500/2,766
Embedded 2,600/2,766
Embedded 2,700/2,766
Embedded 2,766/2,766


## Save clean files

In [7]:
clean_products.to_csv("./Clean CSV/products_clean.csv", index=False, encoding="utf-8")
variants.to_csv("./Clean CSV/variants_clean.csv", index=False, encoding="utf-8")
images.to_csv("./Clean CSV/images_clean.csv", index=False, encoding="utf-8")

print(f"Products saved: {len(clean_products):,}")
print(f"Variants saved: {len(variants):,}")
print(f"Images saved: {len(images):,}")
print("Output: ./Clean CSV")

Products saved: 2,766
Variants saved: 16,989
Images saved: 12,978
Output: ./Clean CSV
